# Configuracion Inicial del Proyecto

## 1. Creacion de Catalogo Schema y Volumen

In [0]:
# ============================================
# Creación estructura Unity Catalog
# Proyecto: fintech_finpay
# ============================================

catalog_name = "fintech_finpay"

schemas = [
    "bronze",
    "silver",
    "gold",
    "observability"
]

volume_name = "vol_landing"

# ============================================
# 1. Crear catálogo
# ============================================

spark.sql(f"""
CREATE CATALOG IF NOT EXISTS {catalog_name}
""")

print(f"Catálogo creado/verificado: {catalog_name}")

# ============================================
# 2. Crear schemas
# ============================================

for schema in schemas:
    spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema}
    """)
    
    print(f"Schema creado/verificado: {catalog_name}.{schema}")

# ============================================
# 3. Crear Volume en schema default
# ============================================

spark.sql(f"""
CREATE VOLUME IF NOT EXISTS
{catalog_name}.default.{volume_name}
""")

print(f"""
Volume creado/verificado:
    /Volumes/{catalog_name}/default/{volume_name}/
""")

# ============================================
# 4. Mostrar estructura creada
# ============================================

print("====================================")
print("Estructura creada:")
print("====================================")
print(f"Catalogo: {catalog_name}")

print("\nSchemas:")
for schema in schemas:
    print(f" - {schema}")

print(f"\nLanding zone:")
print(f"/Volumes/{catalog_name}/default/{volume_name}/")


Catálogo creado/verificado: fintech_finpay
Schema creado/verificado: fintech_finpay.default
Schema creado/verificado: fintech_finpay.bronze
Schema creado/verificado: fintech_finpay.silver
Schema creado/verificado: fintech_finpay.gold
Schema creado/verificado: fintech_finpay.observability

Volume creado/verificado:
    /Volumes/fintech_finpay/default/vol_landing/

Estructura creada:
Catalogo: fintech_finpay

Schemas:
 - default
 - bronze
 - silver
 - gold
 - observability

Landing zone:
/Volumes/fintech_finpay/default/vol_landing/


## 2 Creacion de Estructura para Landing zone

In [0]:
# Databricks - creación de estructura Landing Zone

LANDING_ZONE = "/Volumes/fintech_finpay/default/vol_landing"

folders = [
    "metadata",
    "transactions",
    "merchants",
    "users",
    "_schemas",
    "_checkpoints"
]

for folder in folders:
    path = f"{LANDING_ZONE}/{folder}"
    dbutils.fs.mkdirs(path)
    print(f"Creado/verificado: {path}")

Creado/verificado: /Volumes/fintech_finpay/default/vol_landing/metadata
Creado/verificado: /Volumes/fintech_finpay/default/vol_landing/transactions
Creado/verificado: /Volumes/fintech_finpay/default/vol_landing/merchants
Creado/verificado: /Volumes/fintech_finpay/default/vol_landing/users
Creado/verificado: /Volumes/fintech_finpay/default/vol_landing/_schemas
Creado/verificado: /Volumes/fintech_finpay/default/vol_landing/_checkpoints


## 3 Creacion de Grupos

In [0]:
# ============================================
# Creación de grupos/roles de acceso
# Proyecto: fintech_finpay
# ============================================

catalog = "fintech_finpay"

roles = {
    "ingenieria": {
        "schemas": ["bronze", "silver", "gold", "observability"],
        "permissions": ["USE SCHEMA", "CREATE TABLE", "MODIFY"]
    },
    "riesgo": {
        "schemas": ["silver", "gold"],
        "permissions": ["USE SCHEMA", "SELECT"]
    },
    "auditoria": {
        "schemas": ["gold", "observability"],
        "permissions": ["USE SCHEMA", "SELECT"]
    }
}

# ============================================
# Obtener grupos existentes
# ============================================

existing_groups_df = spark.sql("SHOW GROUPS")

existing_groups = [
    row[0]
    for row in existing_groups_df.collect()
]

print("====================================")
print("Grupos existentes:")
print("====================================")

for grp in existing_groups:
    print(f" - {grp}")

# ============================================
# Crear grupos si no existen
# ============================================

print("\n====================================")
print("Creando grupos faltantes")
print("====================================")

for role in roles.keys():

    if role in existing_groups:

        print(f"Grupo ya existe: {role}")

    else:

        try:

            spark.sql(f"""
            CREATE GROUP `{role}`
            """)

            print(f"Grupo creado: {role}")

        except Exception as e:

            print(f"Error creando grupo {role}")
            print(str(e))

# ============================================
# Asignar USE CATALOG
# ============================================

print("\n====================================")
print("Asignando USE CATALOG")
print("====================================")

for role in roles.keys():

    try:

        sql_use= f"""
        GRANT USE CATALOG
        ON CATALOG `{catalog}`
        TO `{role}`
        """
        spark.sql(sql_use)

        print(f"USE CATALOG otorgado a {role}")

    except Exception as e:

        print(f"Error asignando USE CATALOG a {role}")
        print(f"Sentencia usada {sql_use}")
        print(str(e))
# ============================================
# Asignar permisos por schema
# ============================================

print("\n====================================")
print("Asignando permisos por schema")
print("====================================")

for role, config in roles.items():

    for schema in config["schemas"]:

        for permission in config["permissions"]:

            try:

                spark.sql(f"""
                GRANT {permission}
                ON SCHEMA `{catalog}`.`{schema}`
                TO `{role}`
                """)

                print(
                    f"{permission} otorgado sobre "
                    f"{catalog}.{schema} a {role}"
                )

            except Exception as e:

                print(
                    f"Error asignando {permission} "
                    f"sobre {catalog}.{schema} a {role}"
                )

                print(str(e))

print("\n====================================")
print("Proceso finalizado.")
print("====================================")

## 5 Generar el achivo configuracion base 

In [ ]:
# ============================================
# Imports
# ============================================

import json

# ============================================
# Variables
# ============================================

LANDING_ZONE = "/Volumes/fintech_finpay/default/vol_landing"

SOURCE_FILE_NAME = "ingestion_archetypes.json"

CONFIG_PATH = f"{LANDING_ZONE}/metadata/{SOURCE_FILE_NAME}"


# ============================================
# Obtener path relativo al notebook
# ============================================

notebook_path = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)

notebook_dir = notebook_path.rsplit("/", 1)[0]

workspace_source_path = f"/Workspace{notebook_dir}/../tmp/template_{SOURCE_FILE_NAME}"

# ============================================
# Leer archivo JSON original
# ============================================

with open(workspace_source_path, "r", encoding="utf-8") as f:
    json_content = json.load(f)

# ============================================
# Escribir JSON en Volume
# ============================================

dbutils.fs.put(
    CONFIG_PATH,
    json.dumps(json_content, indent=4),
    True
)

print(f"Archivo JSON generado en: {CONFIG_PATH}")

## 6. copiar los datos

In [ ]:
# ============================================
# Copiar data demo desde tmp hacia vol_landing
# Proyecto: fintech_finpay
# ============================================

import os

# ============================================
# Variables
# ============================================

LANDING_ZONE = "/Volumes/fintech_finpay/default/vol_landing"

SOURCE_BASE_FOLDER = "data_demo"

folders = {
    "users": "txt",
    "transactions": "csv",
    "merchants": "json"
}

# ============================================
# Obtener path relativo al notebook
# ============================================

notebook_path = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)

notebook_dir = notebook_path.rsplit("/", 1)[0]

source_base_path = f"file:/Workspace{notebook_dir}/../tmp/{SOURCE_BASE_FOLDER}"

# ============================================
# Crear carpetas destino si no existen
# ============================================

for folder_name in folders.keys():
    target_path = f"{LANDING_ZONE}/{folder_name}"
    dbutils.fs.mkdirs(target_path)
    print(f"Carpeta destino verificada: {target_path}")

# ============================================
# Copiar carpetas y contenidos
# ============================================

for folder_name, extension in folders.items():

    source_path = f"{source_base_path}/{folder_name}"
    target_path = f"{LANDING_ZONE}/{folder_name}"

    print("====================================")
    print(f"Copiando carpeta: {folder_name}")
    print(f"Origen : {source_path}")
    print(f"Destino: {target_path}")
    print(f"Formato esperado: .{extension}")
    print("====================================")

    files = dbutils.fs.ls(source_path)

    for file in files:

        if file.isDir():
            continue

        if not file.name.lower().endswith(f".{extension}"):
            print(f"Archivo omitido por extensión no válida: {file.name}")
            continue

        source_file = file.path
        target_file = f"{target_path}/{file.name}"

        dbutils.fs.cp(
            source_file,
            target_file,
            recurse=False
        )

        print(f"Archivo copiado: {file.name}")

print("====================================")
print("Copia de data_demo finalizada.")
print("====================================")